In [29]:
import numpy as np
import pandas as pd

teams = pd.read_csv(" Fifa_WC_2026/master_team_features.csv")

# Index by team
teams_dict = teams.set_index("team").to_dict("index")


def simulate_match(team_a, team_b, knockout=False):

    A = teams_dict[team_a]
    B = teams_dict[team_b]

    
    # Win strength
    
    strength_a = (
        (A["current_elo"] * 0.70) +
        (A["form_score"] * 300 * 0.15) +
        (A["team_strength_score"] * 200 * 0.15) +
        (A["knockout_win_rate"] * 100 * 0.05) +
        (A["deep_run_score"] * 10 * 0.05)
    )

    strength_b = (
        (B["current_elo"] * 0.70) +
        (B["form_score"] * 300 * 0.15) +
        (B["team_strength_score"] * 200 * 0.15) +
        (B["knockout_win_rate"] * 100 * 0.05) +
        (B["deep_run_score"] * 10 * 0.05)
    )

    elo_diff = strength_a - strength_b
    if knockout:
        elo_diff += np.random.normal(0, 20)
    else:
        elo_diff += np.random.normal(0, 35)
    
    # Goal expectation
    
    attack_a = A["goals_for_avg_last20"]
    attack_b = B["goals_for_avg_last20"]

    defense_a = B["goals_against_avg_last20"]
    defense_b = A["goals_against_avg_last20"]

    clean_adj_a = (1 - B["clean_sheet_rate"] * 0.30)
    clean_adj_b = (1 - A["clean_sheet_rate"] * 0.30)

    
    lambda_a = ((attack_a + defense_a) / 2) * clean_adj_a
    lambda_b = ((attack_b + defense_b) / 2) * clean_adj_b

    # Elo impact (small)
    elo_boost = np.tanh(elo_diff / 400)

    lambda_a *= (1 + elo_boost * 0.25)
    lambda_b *= (1 - elo_boost * 0.25)

    # floor protection
    lambda_a = max(lambda_a, 0.2)
    lambda_b = max(lambda_b, 0.2)

    
    # 3. Simulate FT
    
    goals_a = np.random.poisson(lambda_a)
    goals_b = np.random.poisson(lambda_b)

    result = {
        "team_a": team_a,
        "team_b": team_b,
        "goals_a": goals_a,
        "goals_b": goals_b,
        "winner": None,
        "method": "FT"
    }

    
    # Group stage
    
    if not knockout:
        if goals_a > goals_b:
            result["winner"] = team_a
        elif goals_b > goals_a:
            result["winner"] = team_b
        else:
            result["winner"] = "Draw"

        return result

    
    # Knockout stage
    
    if goals_a > goals_b:
        result["winner"] = team_a
        return result

    if goals_b > goals_a:
        result["winner"] = team_b
        return result

    # Extra time (lower intensity)
    et_a = np.random.poisson(lambda_a * 0.30)
    et_b = np.random.poisson(lambda_b * 0.30)

    goals_a += et_a
    goals_b += et_b

    if goals_a > goals_b:
        result["winner"] = team_a
        result["method"] = "ET"
        result["goals_a"] = goals_a
        result["goals_b"] = goals_b
        return result

    if goals_b > goals_a:
        result["winner"] = team_b
        result["method"] = "ET"
        result["goals_a"] = goals_a
        result["goals_b"] = goals_b
        return result

    # Penalties
    pen_a = A["penalty_strength"] + (A["penalty_experience"] * 0.05)
    pen_b = B["penalty_strength"] + (B["penalty_experience"] * 0.05)

    total = pen_a + pen_b

    if total == 0:
        pen_prob_a = 0.5
    else:
        pen_prob_a = pen_a / total

    winner = team_a if np.random.rand() < pen_prob_a else team_b

    result["winner"] = winner
    result["method"] = "PEN"
    result["goals_a"] = goals_a
    result["goals_b"] = goals_b

    return result

# "Fixtures"

In [30]:
fixtures = pd.read_csv(" Fifa_WC_2026/fixtures_wc26.csv")

# Only group-stage fixtures
group_fixtures = fixtures[fixtures["stage"] == "Group"].copy()


def initialize_table(teams):

    table = {}

    for team in teams:
        table[team] = {
            "MP": 0,
            "W": 0,
            "D": 0,
            "L": 0,
            "GF": 0,
            "GA": 0,
            "GD": 0,
            "Pts": 0
        }

    return table


def update_table(table, result):

    teamA = result["team_a"]
    teamB = result["team_b"]
    goalsA = result["goals_a"]
    goalsB = result["goals_b"]

    table[teamA]["MP"] += 1
    table[teamB]["MP"] += 1

    table[teamA]["GF"] += goalsA
    table[teamA]["GA"] += goalsB

    table[teamB]["GF"] += goalsB
    table[teamB]["GA"] += goalsA

    table[teamA]["GD"] = table[teamA]["GF"] - table[teamA]["GA"]
    table[teamB]["GD"] = table[teamB]["GF"] - table[teamB]["GA"]

    if goalsA > goalsB:
        table[teamA]["W"] += 1
        table[teamB]["L"] += 1
        table[teamA]["Pts"] += 3

    elif goalsB > goalsA:
        table[teamB]["W"] += 1
        table[teamA]["L"] += 1
        table[teamB]["Pts"] += 3

    else:
        table[teamA]["D"] += 1
        table[teamB]["D"] += 1
        table[teamA]["Pts"] += 1
        table[teamB]["Pts"] += 1


def simulate_group(group_name):

    group_matches = group_fixtures[group_fixtures["group"] == group_name]

    teams = pd.unique(
        pd.concat([
            group_matches["home_team"],
            group_matches["away_team"]
        ])
    )

    table = initialize_table(teams)

    results = []

    for _, match in group_matches.iterrows():

        result = simulate_match(
            match["home_team"],
            match["away_team"],
            knockout=False
        )

        results.append(result)
        update_table(table, result)

    table_df = pd.DataFrame(table).T

    table_df = table_df.sort_values(
        by=["Pts", "GD", "GF"],
        ascending=False
    )

    table_df = table_df.reset_index().rename(columns={"index": "Team"})
    table_df["Position"] = range(1, len(table_df)+1)

    return results, table_df


def simulate_full_group_stage():

    all_results = {}
    all_tables = {}
    qualifiers = {}

    groups = sorted(group_fixtures["group"].unique())

    for group in groups:

        results, table = simulate_group(group)

        all_results[group] = results
        all_tables[group] = table

        qualifiers[group] = {
        "1st": table.iloc[0]["Team"],
        "2nd": table.iloc[1]["Team"]
        }

    return all_results, all_tables, qualifiers
results, tables, qualifiers = simulate_full_group_stage()

# "Top 8 Third Place and Knockout bracket"

In [31]:
# Best third-place teams
def get_best_third_place_teams(all_tables):

    third_place_rows = []

    for group, table in all_tables.items():
        third = table.iloc[2].copy()
        third["Group"] = group
        third_place_rows.append(third)

    third_df = pd.DataFrame(third_place_rows)

    third_df = third_df.sort_values(
        by=["Pts", "GD", "GF"],
        ascending=False
    )

    return third_df.head(8)


def build_round_of_32(qualifiers, all_tables):

    best_8_thirds = get_best_third_place_teams(all_tables)

    third_lookup = {
        row["Group"]: row["Team"]
        for _, row in best_8_thirds.iterrows()
    }

    winner_order = ["A", "B", "C", "E", "F", "I", "J", "K"]

    dynamic_alloc = {}

    available_groups = list(third_lookup.keys())
    np.random.shuffle(available_groups)

    for winner_group in winner_order:

        valid_groups = [
            g for g in available_groups
            if g != winner_group
        ]

        # fallback if dead-end happens
        if len(valid_groups) == 0:

            for prev_group in dynamic_alloc:

                old_team = dynamic_alloc[prev_group]

                old_group = None
                for g, team in third_lookup.items():
                    if team == old_team:
                        old_group = g
                        break

                if old_group != winner_group and prev_group != winner_group:
                    dynamic_alloc[prev_group] = third_lookup[winner_group]
                    dynamic_alloc[winner_group] = old_team
                    break

            continue

        chosen_group = np.random.choice(valid_groups)

        dynamic_alloc[winner_group] = third_lookup[chosen_group]

        available_groups.remove(chosen_group)

    round_of_32 = [

        # Match 73
        (qualifiers["A"]["2nd"], qualifiers["B"]["2nd"]),

        # Match 74
        (qualifiers["E"]["1st"], dynamic_alloc["E"]),

        # Match 75
        (qualifiers["F"]["1st"], qualifiers["C"]["2nd"]),

        # Match 76
        (qualifiers["C"]["1st"], qualifiers["F"]["2nd"]),

        # Match 77
        (qualifiers["I"]["1st"], dynamic_alloc["I"]),

        # Match 78
        (qualifiers["E"]["2nd"], qualifiers["I"]["2nd"]),

        # Match 79
        (qualifiers["A"]["1st"], dynamic_alloc["A"]),

        # Match 80
        (qualifiers["L"]["1st"], dynamic_alloc["K"]),

        # Match 81
        (qualifiers["D"]["1st"], dynamic_alloc["B"]),

        # Match 82
        (qualifiers["G"]["1st"], dynamic_alloc["C"]),

        # Match 83
        (qualifiers["K"]["2nd"], qualifiers["L"]["2nd"]),

        # Match 84
        (qualifiers["H"]["1st"], qualifiers["J"]["2nd"]),

        # Match 85
        (qualifiers["B"]["1st"], dynamic_alloc["F"]),

        # Match 86
        (qualifiers["J"]["1st"], qualifiers["H"]["2nd"]),

        # Match 87
        (qualifiers["K"]["1st"], dynamic_alloc["J"]),

        # Match 88
        (qualifiers["D"]["2nd"], qualifiers["G"]["2nd"]),
    ]

    return round_of_32


round_of_32 = build_round_of_32(qualifiers, tables)

# "Knockouts"

In [32]:

# ROUND OF 16 (89–96)

def build_round_of_16(r32_winners):

    return [
        # Match 89
        (r32_winners[1], r32_winners[4]),

        # Match 90
        (r32_winners[0], r32_winners[2]),

        # Match 91
        (r32_winners[3], r32_winners[5]),

        # Match 92
        (r32_winners[6], r32_winners[7]),

        # Match 93
        (r32_winners[10], r32_winners[11]),

        # Match 94
        (r32_winners[8], r32_winners[9]),

        # Match 95
        (r32_winners[13], r32_winners[15]),

        # Match 96
        (r32_winners[12], r32_winners[14]),
    ]


# QUARTER FINALS (97–100)

def build_quarter_finals(r16_winners):

    return [
        # Match 97
        (r16_winners[0], r16_winners[1]),

        # Match 98
        (r16_winners[5], r16_winners[4]),

        # Match 99
        (r16_winners[2], r16_winners[3]),

        # Match 100
        (r16_winners[6], r16_winners[7]),
    ]



# SEMI FINALS (101–102)

def build_semi_finals(qf_winners):

    return [
        # Match 101
        (qf_winners[0], qf_winners[1]),

        # Match 102
        (qf_winners[2], qf_winners[3]),
    ]


# FINAL (104)

def build_final(sf_winners):

    return [
        # Match 104
        (sf_winners[0], sf_winners[1])
    ]

# "WC Simulation"

In [33]:
def simulate_round(fixtures, start_match_no=0):

    results = []
    winners = []

    for i, match in enumerate(fixtures):

        team_a = match[0]
        team_b = match[1]

        result = simulate_match(
            team_a,
            team_b,
            knockout=True
        )

        results.append(result)
        winners.append(result["winner"])

        ''' optional print
        if start_match_no:
            print(
                f"Match {start_match_no+i}: "
                f"{team_a} {result['goals_a']} - {result['goals_b']} {team_b} "
                f"({result['winner']} via {result['method']})"
            )'''

    return results, winners

def simulate_tournament():

    # Group stage
    group_results, tables, qualifiers = simulate_full_group_stage()

    # Round of 32
    round_of_32 = build_round_of_32(qualifiers, tables)
    r32_results, r32_winners = simulate_round(round_of_32, 73)

    # Round of 16
    round_of_16 = build_round_of_16(r32_winners)
    r16_results, r16_winners = simulate_round(round_of_16, 89)

    # Quarterfinals
    quarterfinals = build_quarter_finals(r16_winners)
    qf_results, qf_winners = simulate_round(quarterfinals, 97)

    # Semifinals
    semifinals = build_semi_finals(qf_winners)
    sf_results, sf_winners = simulate_round(semifinals, 101)

    # Final
    final = build_final(sf_winners)
    final_results, final_winners = simulate_round(final, 104)

    champion = final_winners[0]

    return {
        "tables": tables,
        "qualifiers": qualifiers,
        "champion": champion
    }

wc = simulate_tournament()

# "Monte Carlo"

In [35]:
def monte_carlo(n=1000):

    winners = {}

    for team in teams["team"]:
        winners[team] = 0

    for _ in range(n):

        tournament = simulate_tournament()
        champion = tournament["champion"]

        winners[champion] += 1

    results = pd.DataFrame.from_dict(
        winners,
        orient="index",
        columns=["Winner %"]
    )

    results["Winner %"] = (results["Winner %"] / n) * 100

    results = results.sort_values(
        by="Winner %",
        ascending=False
    )

    return results


mc_results = monte_carlo(50000)
print(mc_results)

                        Winner %
Argentina                   13.4
England                     12.5
Morocco                     10.9
Japan                        9.5
Spain                        6.5
Norway                       4.4
Algeria                      4.1
Croatia                      3.2
Senegal                      3.2
Germany                      2.9
France                       2.7
Netherlands                  2.4
Brazil                       2.1
Portugal                     1.8
Ivory Coast                  1.8
Colombia                     1.6
Austria                      1.6
Belgium                      1.5
Iran                         1.4
Canada                       1.3
Turkey                       1.1
Mexico                       1.0
Uzbekistan                   0.9
South Korea                  0.9
Egypt                        0.9
Paraguay                     0.7
Czech Republic               0.7
Ecuador                      0.6
Australia                    0.5
Uruguay   